# PT Workflow Orchestrator
Führt den vollständigen Workflow aus:
**PT_getData → PT_Predictor_live → PT_Vorarbeiten → PT_Create_HTMLs_fast**

Alle Notebooks werden via `papermill` sequenziell ausgeführt.
Execution logs werden in `{BASE_PATH}/workflow_logs/` gespeichert.

In [ ]:
# ── 1. SETUP ────────────────────────────────────────────────────────────────
!pip install -q papermill

from google.colab import drive
drive.mount('/content/drive')

import papermill as pm
import json, time, os
from datetime import datetime
import pytz

BERLIN = pytz.timezone('Europe/Berlin')
TODAY  = datetime.now(BERLIN).strftime('%Y-%m-%d')

# ── Pfade ───────────────────────────────────────────────────────────────────
BASE_PATH  = '/content/drive/MyDrive/PT'
REPO_PATH  = '/content/ptnew'          # Wo das Repo geklont wird
LOG_PATH   = f'{BASE_PATH}/workflow_logs'

os.makedirs(LOG_PATH, exist_ok=True)

print(f'Datum       : {TODAY}')
print(f'Repo        : {REPO_PATH}')
print(f'Logs        : {LOG_PATH}')
# ── Secrets (fetched here, passed to sub-notebooks as parameters) ───────────
from google.colab import userdata as _ud
ANTHROPIC_API_KEY = _ud.get('ANTHROPIC_API_KEY')
GITHUB_TOKEN      = _ud.get('GITHUB_TOKEN')
print(f'Secrets geladen: ANTHROPIC_API_KEY={"✓" if ANTHROPIC_API_KEY else "✗"}  '
      f'GITHUB_TOKEN={"✓" if GITHUB_TOKEN else "✗"}')


In [ ]:
# ── 2. REPO HOLEN ───────────────────────────────────────────────────────────
if not os.path.exists(f'{REPO_PATH}/.git'):
    print('Klone Repository ...')
    !git clone https://github.com/arauch6363-crypto/ptnew.git {REPO_PATH}
else:
    print('Aktualisiere Repository ...')
    !git -C {REPO_PATH} pull --ff-only

# Sync repo files to Drive so all notebooks use the latest versions
import shutil
shutil.copy(f'{REPO_PATH}/pt_html_functions.py',  f'{BASE_PATH}/pt_html_functions.py')
shutil.copy(f'{REPO_PATH}/going_overrides.json',   f'{BASE_PATH}/going_overrides.json')

print(f'Notebooks verfügbar unter: {REPO_PATH}')
print(f'✓ pt_html_functions.py nach Drive synchronisiert')
print(f'✓ going_overrides.json nach Drive synchronisiert')


In [ ]:
# ── 3. WORKFLOW-KONFIGURATION ────────────────────────────────────────────────
# Reihenfolge und Timeouts (Sekunden) je Schritt.
# Einen Schritt überspringen: 'skip': True hinzufügen.

WORKFLOW = [
    {
        'name'      : 'PT_getData',
        'notebook'  : f'{REPO_PATH}/PT_getData.ipynb',
        'timeout'   : 1800,
        'skip'      : False,
        'parameters': {},
    },
    {
        'name'      : 'PT_monitor_learning',
        'notebook'  : f'{REPO_PATH}/PT_monitor_learning.ipynb',
        'timeout'   : 600,
        'skip'      : False,
        'parameters': {'ANTHROPIC_API_KEY': ANTHROPIC_API_KEY},
    },
    {
        'name'      : 'PT_Predictor_live',
        'notebook'  : f'{REPO_PATH}/PT_Predictor_live.ipynb',
        'timeout'   : 900,
        'skip'      : False,
        'parameters': {},
    },
    {
        'name'      : 'PT_Vorarbeiten',
        'notebook'  : f'{REPO_PATH}/PT_Vorarbeiten.ipynb',
        'timeout'   : 900,
        'skip'      : False,
        'parameters': {'ANTHROPIC_API_KEY': ANTHROPIC_API_KEY},
    },
    {
        'name'      : 'PT_Create_HTMLs_fast',
        'notebook'  : f'{REPO_PATH}/PT_Create_HTMLs_fast.ipynb',
        'timeout'   : 300,
        'skip'      : False,
        'parameters': {'GITHUB_TOKEN': GITHUB_TOKEN},
    },
]

active_steps = [s for s in WORKFLOW if not s.get('skip')]
print(f'{len(active_steps)} Schritte aktiv:')
for i, s in enumerate(active_steps, 1):
    print(f'  {i}. {s["name"]}  (max {s["timeout"]}s)')


In [ ]:
# ── 4. WORKFLOW AUSFÜHREN ───────────────────────────────────────────────────
results        = []
workflow_start = time.time()
workflow_ok    = True

for step in active_steps:
    step_start = time.time()
    status     = 'SUCCESS'
    error_msg  = None
    output_nb  = f'{LOG_PATH}/{TODAY}_{step["name"]}_out.ipynb'

    print()
    print('=' * 62)
    print(f'  ▶  {step["name"]}')
    print(f'     Start: {datetime.now(BERLIN).strftime("%H:%M:%S")}')
    print('=' * 62)

    try:
        pm.execute_notebook(
            input_path  = step['notebook'],
            output_path = output_nb,
            kernel_name = 'python3',
            timeout     = step['timeout'],
            progress_bar= True,
            parameters  = step.get('parameters', {}),
        )
        elapsed = round(time.time() - step_start)
        print(f'\n  ✓ {step["name"]} abgeschlossen in {elapsed}s')

    except pm.exceptions.PapermillExecutionError as e:
        elapsed     = round(time.time() - step_start)
        status      = 'FAILED'
        workflow_ok = False
        # Knappe Fehlermeldung: nur die letzte Exception-Zeile
        error_lines = str(e).strip().splitlines()
        error_msg   = next(
            (l for l in reversed(error_lines) if l.strip()),
            str(e)[:300]
        )
        print(f'\n  ✗ {step["name"]} FEHLGESCHLAGEN nach {elapsed}s')
        print(f'    Fehler: {error_msg}')
        print(f'    Output-Notebook gespeichert: {output_nb}')

    except Exception as e:
        elapsed     = round(time.time() - step_start)
        status      = 'FAILED'
        workflow_ok = False
        error_msg   = f'{type(e).__name__}: {str(e)[:300]}'
        print(f'\n  ✗ Unerwarteter Fehler in {step["name"]}: {error_msg}')

    results.append({
        'step'           : step['name'],
        'status'         : status,
        'elapsed_s'      : elapsed,
        'output_notebook': output_nb,
        'error'          : error_msg,
    })

    if not workflow_ok:
        print('\n⚠  Workflow abgebrochen.')
        break

# ── Zusammenfassung ──────────────────────────────────────────────────────────
total_s = round(time.time() - workflow_start)

print()
print('=' * 62)
label = '✓ WORKFLOW ERFOLGREICH' if workflow_ok else '✗ WORKFLOW FEHLGESCHLAGEN'
print(f'  {label}  ({total_s}s gesamt)')
print('=' * 62)
for r in results:
    icon = '✓' if r['status'] == 'SUCCESS' else '✗'
    print(f'  {icon}  {r["step"]:35s}  {r["elapsed_s"]:4d}s')
print()

# ── Log auf Drive speichern ──────────────────────────────────────────────────
log = {
    'date'           : TODAY,
    'run_at'         : datetime.now(BERLIN).isoformat(),
    'status'         : 'SUCCESS' if workflow_ok else 'FAILED',
    'total_elapsed_s': total_s,
    'steps'          : results,
}
log_file = f'{LOG_PATH}/{TODAY}_workflow.json'
with open(log_file, 'w', encoding='utf-8') as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

print(f'Log gespeichert: {log_file}')

In [ ]:
# ── 5. (OPTIONAL) LETZTE WORKFLOW-LOGS ANZEIGEN ─────────────────────────────
import glob

logs = sorted(glob.glob(f'{LOG_PATH}/*_workflow.json'))[-5:]
print(f'Letzte {len(logs)} Workflow-Runs:\n')

for log_path in reversed(logs):
    with open(log_path) as f:
        data = json.load(f)
    icon = '✓' if data['status'] == 'SUCCESS' else '✗'
    print(f'{icon}  {data["date"]}  '
          f'({data["total_elapsed_s"]}s)  '
          f'run_at={data["run_at"][11:19]}')
    for s in data['steps']:
        si = '  ✓' if s['status'] == 'SUCCESS' else '  ✗'
        err = f"  → {s['error']}" if s.get('error') else ''
        print(f'{si}  {s["step"]:35s}  {s["elapsed_s"]:4d}s{err}')
    print()